In [ ]:
# 1. Install necessary libraries
!pip install -q -U google-genai pymupdf

# 2. Imports
import os
import json
import time
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata, drive

drive.mount('/content/drive')

# Paths (Ensure these are mounted in Colab)
BASE_DIR = "PATH TO YOUR BASE DIR"
INDEX_DIR = f"{BASE_DIR}/indices"
CONTENT_DIR = f"{BASE_DIR}/raw_pdfs"
TEST_FILE = f"{BASE_DIR}/test/jurismind_100.csv"
RESULTS_FILE = f"{BASE_DIR}/test/jurismind_results_scored.csv"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.2/764.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 12.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.
Mounted at /content/drive


In [ ]:
import os
import json
import time
import pandas as pd
from google import genai

# --- CONFIGURATION ---
api_key = 'API_KEY'
client = genai.Client(api_key=api_key)

# Updated Paths
BASE_DIR = "PATH TO YOUR BASE DIR"
INDEX_DIR = f"{BASE_DIR}/indices"
CONTENT_DIR = f"{BASE_DIR}/content"
TEST_FILE = f"{BASE_DIR}/test/jurismind_100.csv"
RESULTS_FILE = f"{BASE_DIR}/test/jurismind_results_scored.csv"

# --- THE JURISMIND ENGINE ---
def ask_jurismind_batch(query):
    # Dictionary to capture data for the CSV results
    run_log = {"selected_doc": None, "target_pages": [], "response": "ERROR"}

    # --- STAGE 1: THE LIBRARIAN ---
    with open(f"{INDEX_DIR}/catalog.json", "r") as f:
        catalog = json.load(f)
    catalog_context = "\n".join([f"DOC_ID: {k} | {v['routing_metadata']}" for k, v in catalog.items()])

    router_prompt = f"""
    You are an expert Legal Librarian. Analyze the provided DOCUMENT metadata.
    DOCUMENTS:
    {catalog_context}
    QUERY: {query}
    TASK:
    1. Identify the PRIMARY DOMAIN of the query (Patent, Design, or Trademark).
    2. Check the EXCLUSIONS in the metadata. If a document explicitly 'Excludes' the query topic, REJECT IT.
    3. Match the query to the document with the most relevant TECHNICAL ANCHORS and TARGET TRIGGERS.
    Respond ONLY with the single DOC_ID.
    """
    try:
        doc_id = client.models.generate_content(model="gemini-flash-latest", contents=router_prompt).text.strip()
        doc_id = doc_id.replace("DOC_ID:", "").strip()
        run_log["selected_doc"] = doc_id
    except: return run_log

    # --- STAGE 2: THE NAVIGATOR ---
    try:
        with open(f"{INDEX_DIR}/{doc_id}_index.json", "r") as f:
            doc_index = json.load(f)
        index_context = "\n".join([f"Page {k}: {v}" for k, v in doc_index.items()])
        navigator_prompt = f"Identify page numbers for query. INDEX:\n{index_context}\nQUERY: {query}\nRespond ONLY page numbers (e.g. 45, 46)."
        page_resp = client.models.generate_content(model="gemini-flash-latest", contents=navigator_prompt).text.strip()
        target_pages = [p.strip() for p in page_resp.split(",") if p.strip().isdigit()]
        run_log["target_pages"] = target_pages
    except: return run_log

    # --- STAGE 3: THE NEIGHBOR-AWARE FETCHER ---
    final_context_block = ""
    content_dir = f"{CONTENT_DIR}/{doc_id}"

    pages_to_fetch = set()
    for p in target_pages:
        n = int(p)
        pages_to_fetch.update([n-1, n, n+1])
    sorted_pages = sorted([p for p in pages_to_fetch if p > 0])

    for p in sorted_pages:
        path = f"{content_dir}/page_{p}.md"
        # Retry logic for Drive sync latency
        for attempt in range(2):
            if os.path.exists(path):
                with open(path, "r", encoding="utf-8") as f:
                    final_context_block += f"\n--- SOURCE: {doc_id} | PAGE {p} ---\n{f.read()}\n"
                break
            time.sleep(2)

    # --- STAGE 4: THE REASONING AGENT ---
    reasoner_prompt = f"""
    You are 'JurisMind', a Senior Intellectual Property Attorney.
    Answer the query using ONLY the provided source content.
    INSTRUCTIONS:
    1. Reassemble sentences split across page boundaries.
    2. Cite the specific Act (e.g., Designs Act 2000 or Patents Act 1970) and Page Numbers.
    3. If the query is about fees or costs, strictly use the tables provided in the source content.
    4. If the source content does not contain the answer, state that you cannot find it in these specific rules.
    SOURCE CONTENT:
    {final_context_block}
    QUERY: {query}
    """
    success, retries = False, 0
    while not success and retries < 3:
        try:
            response = client.models.generate_content(model="gemini-flash-latest", contents=reasoner_prompt)
            run_log["response"] = response.text
            success = True
        except:
            time.sleep(10)
            retries += 1
    return run_log

# --- THE JUDGE AGENT ---
def judge_score(query, gt_answer, gt_doc, gt_pages, ai_response, ai_doc, ai_pages):
    judge_prompt = f"""
    You are a Fair Legal Evaluator. Compare the AI's response to the Ground Truth.

    GROUND TRUTH:
    - Doc ID: {gt_doc}
    - Pages: {gt_pages}
    - Answer: {gt_answer}

    AI RESPONSE:
    - Doc Selected: {ai_doc}
    - Pages Selected: {ai_pages}
    - Content: {ai_response}

    SCORING CRITERIA (0-100):
    1. Accuracy: Does the answer match the ground truth factually?
    2. Routing: Did it pick the correct Doc ID?
    3. Navigation: Did it find the correct pages?
    4. Quality: Are citations present and formatting clear?

    NOTE: Do not be overly strict on minor linguistic variations. Focus on factual correctness and domain accuracy. Don't deduct points for mentioning extra details.

    Return a JSON object: {{"score": 85, "reason": "..."}}
    """
    try:
        resp = client.models.generate_content(model="gemini-flash-latest", contents=judge_prompt)
        clean_resp = resp.text.replace("```json", "").replace("```", "").strip()
        return json.loads(clean_resp)
    except: return {"score": 0, "reason": "Judge error"}

# --- MAIN EVALUATION LOOP ---
def run_evaluation():
    # Load and setup result sheet
    if os.path.exists(RESULTS_FILE):
        df = pd.read_csv(RESULTS_FILE)
    else:
        df = pd.read_csv(TEST_FILE)
        for col in ['JurisMind_Response', 'JurisMind_Doc', 'JurisMind_Pages', 'JurisMind_Score', 'JurisMind_Band']:
            df[col] = None

    print(f"Processing {len(df)} questions...")

    for index, row in df.iterrows():
        # Resume logic
        if pd.notna(row['JurisMind_Score']): continue

        try:
            # Step 1: Run JurisMind logic
            res = ask_jurismind_batch(row['Question'])

            # Step 2: Judge the output
            judgment = judge_score(row['Question'], row['Ground_Truth_Answer'], row['Doc_ID'], row['Page_Numbers'],
                                   res["response"], res["selected_doc"], res["target_pages"])

            # Step 3: Record Results
            score = judgment['score']
            df.at[index, 'JurisMind_Response'] = res["response"]
            df.at[index, 'JurisMind_Doc'] = res["selected_doc"]
            df.at[index, 'JurisMind_Pages'] = str(res["target_pages"])
            df.at[index, 'JurisMind_Score'] = score
            df.at[index, 'JurisMind_Band'] = "High" if score >= 80 else "Medium" if score >= 40 else "Low"

            # Atomic save: overwrite CSV after every question
            df.to_csv(RESULTS_FILE, index=False)
            print(f" [{index+1}/100] Score: {score} ({df.at[index, 'JurisMind_Band']}) | 💾 Saved")

        except Exception as e:
            if "ResourceExhausted" in str(e) or "429" in str(e):
                print(f"API Exhausted at {index+1}. Swap key and rerun.")
                break
            else:
                print(f"Error at {index+1}: {e}")
                break

        time.sleep(12) # Delay to prevent rate limits

In [ ]:
run_evaluation()

In [ ]:
# 1. Install necessary libraries
!pip install -q -U google-genai pymupdf

# 2. Imports
import os
import json
import time
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata, drive

drive.mount('/content/drive')

BASE_DIR = "PATH TO YOUR BASE DIR"
INDEX_DIR = f"{BASE_DIR}/indices"
CONTENT_DIR = f"{BASE_DIR}/content"
TEST_FILE = f"{BASE_DIR}/test/jurismind_100.csv"
RESULTS_FILE = f"{BASE_DIR}/test/jurismind_results_scored.csv"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.1/786.1 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 13.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.
Mounted at /content/drive


In [ ]:

# --- CONFIGURATION ---
api_key = 'API_KEY'
client = genai.Client(api_key=api_key)

# --- THE JURISMIND ENGINE (Using your original logic) ---
def run_jurismind_pipeline(query):
    results = {"selected_doc": None, "selected_pages": [], "response": "ERROR"}

    # --- STAGE 1: THE LIBRARIAN ---
    with open(f"{INDEX_DIR}/catalog.json", "r") as f:
        catalog = json.load(f)
    catalog_context = "\n".join([f"DOC_ID: {k} | {v['routing_metadata']}" for k, v in catalog.items()])

    router_prompt = f"""
    You are an expert Legal Librarian. Analyze the provided DOCUMENT metadata.
    DOCUMENTS:
    {catalog_context}
    QUERY: {query}
    TASK:
    1. Identify the PRIMARY DOMAIN of the query (Patent, Design, or Trademark).
    2. Check the EXCLUSIONS in the metadata. If a document explicitly 'Excludes' the query topic, REJECT IT.
    3. Match the query to the document with the most relevant TECHNICAL ANCHORS and TARGET TRIGGERS.
    Respond ONLY with the single DOC_ID.
    """
    try:
        doc_id = client.models.generate_content(model="gemini-flash-latest", contents=router_prompt).text.strip()
        doc_id = doc_id.replace("DOC_ID:", "").strip()
        results["selected_doc"] = doc_id
    except: return results

    # --- STAGE 2: THE NAVIGATOR ---
    try:
        with open(f"{INDEX_DIR}/{doc_id}_index.json", "r") as f:
            doc_index = json.load(f)
        index_context = "\n".join([f"Page {k}: {v}" for k, v in doc_index.items()])
        navigator_prompt = f"Identify page numbers for query. INDEX:\n{index_context}\nQUERY: {query}\nRespond ONLY page numbers (e.g. 45, 46)."
        page_resp = client.models.generate_content(model="gemini-flash-latest", contents=navigator_prompt).text.strip()
        target_pages = [p.strip() for p in page_resp.split(",") if p.strip().isdigit()]
        results["selected_pages"] = target_pages
    except: return results

    # --- STAGE 3: THE NEIGHBOR-AWARE FETCHER ---
    final_context_block = ""
    content_dir = f"{CONTENT_DIR}/{doc_id}"
    pages_to_fetch = set()
    for p in target_pages:
        n = int(p)
        pages_to_fetch.update([n-1, n, n+1])
    sorted_pages = sorted([p for p in pages_to_fetch if p > 0])

    for p in sorted_pages:
        path = f"{content_dir}/page_{p}.md"
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                final_context_block += f"\n--- SOURCE: {doc_id} | PAGE {p} ---\n{f.read()}\n"

    # --- STAGE 4: THE REASONING AGENT ---
    reasoner_prompt = f"""
    You are 'JurisMind', a Senior Intellectual Property Attorney.
    Answer the query using ONLY the provided source content.
    INSTRUCTIONS:
    1. Reassemble sentences split across page boundaries.
    2. Cite the specific Act (e.g., Designs Act 2000 or Patents Act 1970) and Page Numbers.
    3. If the query is about fees or costs, strictly use the tables provided in the source content.
    4. If the source content does not contain the answer, state that you cannot find it in these specific rules.
    SOURCE CONTENT:
    {final_context_block}
    QUERY: {query}
    """
    success, retries = False, 0
    while not success and retries < 3:
        try:
            response = client.models.generate_content(model="gemini-flash-latest", contents=reasoner_prompt)
            results["response"] = response.text
            success = True
        except:
            time.sleep(10)
            retries += 1
    return results

# --- THE JUDGE AGENT ---
def judge_score(query, gt_answer, gt_doc, gt_pages, ai_response, ai_doc, ai_pages):
    judge_prompt = f"""
    You are a Fair Legal Evaluator. Compare the AI's response to the Ground Truth.

    GROUND TRUTH:
    - Doc ID: {gt_doc}
    - Pages: {gt_pages}
    - Answer: {gt_answer}

    AI RESPONSE:
    - Doc Selected: {ai_doc}
    - Pages Selected: {ai_pages}
    - Content: {ai_response}

    SCORING CRITERIA (0-100):
    1. Accuracy: Does the answer match the ground truth factually?
    2. Routing: Did it pick the correct Doc ID?
    3. Navigation: Did it find the correct pages?
    4. Quality: Are citations present and formatting clear?

    NOTE: Do not be overly strict on minor linguistic variations. Focus on factual correctness and domain accuracy. Don't deduct points for mentioning extra details.

    Return a JSON object: {{"score": 85, "reason": "..."}}
    """
    try:
        resp = client.models.generate_content(model="gemini-flash-latest", contents=judge_prompt)
        clean_resp = resp.text.replace("```json", "").replace("```", "").strip()
        return json.loads(clean_resp)
    except: return {"score": 0, "reason": "Judge Error"}

# --- MAIN EVALUATION LOOP ---
def run_evaluation():
    if os.path.exists(RESULTS_FILE):
        df = pd.read_csv(RESULTS_FILE)
    else:
        df = pd.read_csv(TEST_FILE)
        for col in ['JurisMind_Response', 'JurisMind_Doc', 'JurisMind_Pages', 'JurisMind_Score', 'JurisMind_Band']:
            df[col] = None

    print(f"Starting evaluation for {len(df)} questions...")

    for index, row in df.iterrows():
        # Resume logic: Skip if already scored
        if pd.notna(row['JurisMind_Score']) and row['JurisMind_Response'] != "ERROR":
            continue

        try:
            # 1. Run Pipeline
            pipeline_output = run_jurismind_pipeline(row['Question'])

            # 2. Judge Pipeline
            judgment = judge_score(row['Question'], row['Ground_Truth_Answer'], row['Doc_ID'], row['Page_Numbers'],
                                   pipeline_output["response"], pipeline_output["selected_doc"], pipeline_output["selected_pages"])

            # 3. Apply Banding
            score = judgment['score']
            band = "High" if score >= 80 else "Medium" if score >= 40 else "Low"

            # 4. Save to DF
            df.at[index, 'JurisMind_Response'] = pipeline_output["response"]
            df.at[index, 'JurisMind_Doc'] = pipeline_output["selected_doc"]
            df.at[index, 'JurisMind_Pages'] = str(pipeline_output["selected_pages"])
            df.at[index, 'JurisMind_Score'] = score
            df.at[index, 'JurisMind_Band'] = band

            # ATOMIC SAVE
            df.to_csv(RESULTS_FILE, index=False)
            print(f" [{index+1}/100] Score: {score} ({band}) | Doc: {pipeline_output['selected_doc']} | 💾 Saved")

        except Exception as e:
            if "ResourceExhausted" in str(e) or "429" in str(e):
                print(f"\n API Key Exhausted at question {index+1}. Swap key and rerun.")
                break
            else:
                print(f" Unexpected Error at question {index+1}: {e}")
                break

        time.sleep(12) # Rate limit protection

    print(f"\nProgress Status: {df['JurisMind_Score'].notna().sum()}/{len(df)} questions scored.")

In [ ]:

# --- CONFIGURATION ---
api_key = 'API_KEY'
client = genai.Client(api_key=api_key)

def run_evaluation_retry(target_rows=None):
    if os.path.exists(RESULTS_FILE):
        df = pd.read_csv(RESULTS_FILE)
    else:
        df = pd.read_csv(TEST_FILE)
        for col in ['JurisMind_Response', 'JurisMind_Doc', 'JurisMind_Pages', 'JurisMind_Score', 'JurisMind_Band']:
            df[col] = None

    # Convert row numbers → indices
    target_indices = None
    if target_rows:
        target_indices = {row - 2 for row in target_rows}

    print(f" Starting evaluation for {len(df)} questions...")

    for index, row in df.iterrows():

        # Filter by your CSV row numbers
        if target_indices and index not in target_indices:
            continue

        try:
            pipeline_output = run_jurismind_pipeline(row['Question'])

            judgment = judge_score(
                row['Question'],
                row['Ground_Truth_Answer'],
                row['Doc_ID'],
                row['Page_Numbers'],
                pipeline_output["response"],
                pipeline_output["selected_doc"],
                pipeline_output["selected_pages"]
            )

            score = judgment['score']
            band = "High" if score >= 80 else "Medium" if score >= 40 else "Low"

            df.at[index, 'JurisMind_Response'] = pipeline_output["response"]
            df.at[index, 'JurisMind_Doc'] = pipeline_output["selected_doc"]
            df.at[index, 'JurisMind_Pages'] = str(pipeline_output["selected_pages"])
            df.at[index, 'JurisMind_Score'] = score
            df.at[index, 'JurisMind_Band'] = band

            df.to_csv(RESULTS_FILE, index=False)

            print(f" [Row {index+2}] Score: {score} ({band}) | Saved")

        except Exception as e:
            if "ResourceExhausted" in str(e) or "429" in str(e):
                print(f"\nAPI Key Exhausted at row {index+2}. Swap key and rerun.")
                break
            else:
                print(f" Error at row {index+2}: {e}")
                break

        time.sleep(12)

    print("\nDone re-evaluating selected rows.")

In [ ]:
questions_to_rerun = [35, 85, 88]

run_evaluation_retry(target_rows=questions_to_rerun)